# Exploratory Data Analysis: Part 3 - Filling Missing Values

### Data Cleaning and Imputation Based on Previous EDA

Based on the exploratory data analysis conducted in the first two parts, we have established the following:

- The `responder` variable contains no missing values.
- Features exhibit varying degrees of missingness.

#### Data Cleaning and Imputation

Given that deep learning models require datasets without missing values, this chapter focuses on cleaning and imputing the original data. Our objective is to produce a complete dataset devoid of any nulls, which will then be stored as a `.parquet` file for subsequent modeling stages.

To achieve this, we will employ appropriate imputation strategies tailored to the nature of each feature's missing data. This process ensures that the integrity of the data is maintained while preparing it for use in deep learning algorithms.

By addressing the missing values in the features, we aim to provide a robust dataset that can effectively support the training and evaluation of our predictive models.

In [4]:
# Author: Xinlei Hao
import gc
import os
import polars as pl
from typing import List
from time import time
import logging
import warnings

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

In [2]:
# initial setup
class CONFIG:
    seed = 666
    path = "../data/" # path to the data folder
    save_path = 'result/' # path to save the result
    start_date = 1600 # set what date to start cleaning and imputing


class SpecialCols:
    date_id = "date_id"
    time_id = "time_id"
    symbol_id = "symbol_id"
    weight_col = "weight"
    id_cols = [date_id, time_id, symbol_id]
    target_col = "responder_6"
    target_cols = ["responder_%d" % i for i in range(9)]


### Next Steps: Creating the `DataProcessor` Class

We will now create a `DataProcessor` class that includes three methods for handling missing values:

- **xs_symbol_id**: Imputes missing values using the mean of the feature across other `symbol_id`s at the same `time_id`. This method fills in missing values with the cross-sectional average, ensuring that each `time_id` has consistent data based on contemporaneous observations from other symbols.
  
- **ffill**: Imputes missing values using the value from the previous `time_id` within each `symbol_id` group. This approach leverages forward filling within the time series context, propagating the last observed non-null value forward until another non-null value is encountered.

- **whole_ave**: Imputes missing values using the global mean of each feature across the entire dataset. This method fills in missing values with the overall average of the respective feature, maintaining the general distribution and integrity of the data.



In [3]:
class DataProcessor:
    '''
    class for data processing, including 3 methods to fill nan values
    '''
    def __init__(self, ts_window=100, start_date=678) -> None:
        logger.info("data preparing")
        self._data_path = os.path.join(CONFIG.path, "train.parquet/**/*.parquet")
        self._data = (
            pl.read_parquet(self._data_path)  # read_parquet
            .filter(pl.col(SpecialCols.date_id) >= start_date) # select the data after 677 days (default)
            .with_columns([pl.col(SpecialCols.id_cols)])  # set index
        )

        self._ts_window = ts_window

        print(f'the number of null is:{sum(self._data.null_count().row(0))}')

        gc.collect()
        logger.info("start computing")

    def xs_symbol_id(self, feats: List[str] = None) -> pl.DataFrame:
        """
        Imputes missing values for features in the DataFrame.
        
        For each `symbol_id` and `feature`, if there's a `NaN` value at a specific `time_id`,
        it fills the missing value with the mean of that feature across all other `symbol_id`s at the same `time_id`.
            
        Returns:
            pl.DataFrame: A DataFrame with imputed values for missing entries.
    """
        # extract the required columns, if feats is None, copy the entire data
        data = (
            self._data.select(SpecialCols.id_cols+[SpecialCols.weight_col] + feats + SpecialCols.target_cols) # polars does not have index, need to select
            if feats is not None
            else self._data.clone()
        )
        # get the feature columns
        feat_cols = [col for col in data.columns if col not in (*SpecialCols.id_cols,SpecialCols.weight_col, *SpecialCols.target_cols)] # *SpecialCols.id_cols unpacked as individual parameters

        # grouped by date_id and time_id, then calculate the mean for each time_id of each feature
        # if the outcome of mean is NaN, replace it with null; because polars will treat NaN as float, null_count will not detect it
        grouped_means = data.group_by([SpecialCols.date_id, SpecialCols.time_id]).agg([
            pl.col(col).mean().fill_nan(None).alias(f"{col}_mean")
            for col in feat_cols
        ]).sort([SpecialCols.date_id, SpecialCols.time_id])

        # merge the mean back to the original DataFrame
        means_df = grouped_means.join(data, on=[SpecialCols.date_id, SpecialCols.time_id], how="left")

        # fill the NaN with the corresponding mean, then drop the mean
        means_df = means_df.with_columns([
            pl.when(pl.col(col).is_null()).then(pl.col(f"{col}_mean")).otherwise(pl.col(col)).alias(
                col)
            for col in feat_cols
        ]).drop([col for col in means_df.columns if col.endswith('_mean')])

        self._data = means_df
        gc.collect()
        logger.info("finish xs_symbol_id")
        print(f'the number of null after xs_symbol_id is:{sum(self._data.null_count().row(0))}')
        return self

    def ffill(self, feats: List[str] = None) -> pl.DataFrame:
        """
        Imputes missing values for feature columns using forward fill within each symbol_id group.
        
        For each `symbol_id` and `feature`, if there's a `NaN` value at a specific `time_id`,
        it fills the missing value with the value from the previous `time_id`.
        
        The DataFrame is first grouped by `symbol_id`, then sorted within each group by `date_id` and `time_id`
        in sequential order before applying forward fill.
          
        Returns:
            pl.DataFrame: A DataFrame with missing values imputed using forward fill for each `symbol_id`.
        """
        data = (
            self._data.select(SpecialCols.id_cols+[SpecialCols.weight_col] + feats + SpecialCols.target_cols) 
            if feats is not None
            else self._data.clone()
        )
 
        feat_cols = [col for col in data.columns if col not in (*SpecialCols.id_cols,SpecialCols.weight_col, *SpecialCols.target_cols)] 

        # sort DataFrame to ensure the order is not messed up
        data = data.sort(by=[SpecialCols.symbol_id, SpecialCols.date_id, SpecialCols.time_id])

        # here group by symbol_id
        data = data.with_columns([
            pl.col(col)
            .fill_null(strategy="forward")
            .over(pl.col(SpecialCols.symbol_id)).alias(col)
            for col in feat_cols
        ])

        # sort the DataFrame back to the original order
        self._data = data.sort(by=[*SpecialCols.id_cols])

        gc.collect()
        logger.info("finish ffill")
        print(f'the number of null after ffill is:{sum(self._data.null_count().row(0))}')
        return self

    def whole_ave(self, feats: List[str] = None) -> pl.DataFrame:
        """
        Imputes missing values in feature columns using the global mean of each feature.
        
        For each feature column containing missing values (`NaN`), this function calculates 
        the global mean across all entries for that feature and fills the missing values 
        with this mean. This method helps maintain the overall distribution of the data 
        while addressing missingness.
            
        Returns:
            pl.DataFrame: A DataFrame with missing values imputed using the global mean for each feature.
        """
        data = (
            self._data.select(SpecialCols.id_cols+[SpecialCols.weight_col] + feats + SpecialCols.target_cols) 
            if feats is not None
            else self._data.clone()
        )

        feat_cols = [col for col in data.columns if col not in (*SpecialCols.id_cols,SpecialCols.weight_col, *SpecialCols.target_cols)] 
        mean_values = data.select(feat_cols).mean()

        # use the global mean to fill the null
        self._data = data.with_columns([
            pl.when(pl.col(col).is_null()).then(pl.lit(mean_values[col].item())).otherwise(pl.col(col)).alias(col)
            for col in feat_cols
        ])

        gc.collect()
        logger.info("finish whole_ave")
        print(f'the number of null after whole_ave is:{sum(self._data.null_count().row(0))}')
        return self


### Sequentially Processing Data with `DataProcessor` and Saving as Parquet

#### Objective

To sequentially apply the data cleaning methods provided by the `DataProcessor` class and save the cleaned dataset in `.parquet` format within a specified directory named `result`.


In [5]:
t_start = time()
# create DataProcessor instance
dp = DataProcessor(start_date=CONFIG.start_date) # here we set start_date as 1600

# continuous call methods
dp = dp.xs_symbol_id().ffill().whole_ave()

data_processed = dp._data

# save the processed data as parquet file
data_processed.write_parquet(
    os.path.join(CONFIG.save_path, f"Processed_data_{CONFIG.start_date}.parquet")
)
t_end = time()
print(f'Finish filling null, the number of null is:{sum(data_processed.null_count().row(0))} - Time count{t_end - t_start},Processed_data_{CONFIG.start_date}.parquet has been saved')

INFO:root:data preparing
INFO:root:start computing


the number of null is:1954072


INFO:root:finish xs_symbol_id


the number of null after xs_symbol_id is:1630379


INFO:root:finish ffill
INFO:root:finish whole_ave


the number of null after ffill is:16263
the number of null after whole_ave is:0
Finish filling null, the number of null is:0 - Time count7.594521999359131,Processed_data_1600.parquet has been saved
